In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'/Users/sameekshapandey/HealthMateAI'

In [3]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

/opt/anaconda3/envs/HealthMateAI/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob = "*.pdf",
        loader_cls= PyPDFLoader
    )
    documents = loader.load()
    return documents

In [5]:
extracted_data = load_pdf_files("data")

In [6]:
len(extracted_data)

637

In [7]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs : List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata = {"source": src}
            )
        )
    return minimal_docs

In [8]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [9]:
# Chunking

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [10]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 5859


In [11]:
# Download and return the HuggingFace embeddings model

from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embeddings = download_embeddings()

/var/folders/fd/3vhh3_8d2ml_7sn1hcjj35dh0000gn/T/ipykernel_3061/702002968.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [12]:
vector = embeddings.embed_query("Hello world")
vector

[-0.03447727486491203,
 0.031023193150758743,
 0.0067349509336054325,
 0.02610902115702629,
 -0.03936202451586723,
 -0.16030247509479523,
 0.06692397594451904,
 -0.006441441830247641,
 -0.047450508922338486,
 0.014758889563381672,
 0.0708753690123558,
 0.055527571588754654,
 0.019193297252058983,
 -0.026251325383782387,
 -0.010109535418450832,
 -0.02694052830338478,
 0.022307414561510086,
 -0.022226665169000626,
 -0.14969263970851898,
 -0.017493080347776413,
 0.007676208857446909,
 0.05435230955481529,
 0.00325446343049407,
 0.031725987792015076,
 -0.08462140709161758,
 -0.029405977576971054,
 0.051595646888017654,
 0.04812400043010712,
 -0.003314803121611476,
 -0.05827917903661728,
 0.04196926951408386,
 0.02221072092652321,
 0.12818880379199982,
 -0.022338904440402985,
 -0.011656290851533413,
 0.06292837858200073,
 -0.03287632018327713,
 -0.09122605621814728,
 -0.031175265088677406,
 0.052699532359838486,
 0.04703483730554581,
 -0.0842030718922615,
 -0.030056176707148552,
 -0.0207448

In [13]:
print(f"Vector length: {len(vector)}")

Vector length: 384


In [14]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [15]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [16]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [17]:
pc

In [19]:
from pinecone import ServerlessSpec 

index_name = "health-mate-ai"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [ ]:
# To store our vectors

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents = texts_chunk,
    embedding = embeddings,
    index_name = index_name
)

In [21]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [22]:
# To add more data to the existing Pinecone index

In [23]:
dswith = Document(
    page_content="PCOD is now named as PMOD universally since it was a metabolic disorder and not a ovarian disorder",
    metadata={"source": "Google"}
)

docsearch.add_documents(documents=[dswith])

['5221d15f-9070-4e4a-8bcc-433cc5767133']

In [24]:
# Storage done, now we will create the Retriever

In [25]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [26]:
#test
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='127c38f2-4e58-4c1e-980d-379ed22d6a6a', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='17f36e9f-980e-4b7a-ad63-5032277019eb', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='1b64819c-fee0-4a84-9ce8-146d38382b6e', metadata={'source': 'data/Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with 

In [27]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")

In [28]:

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [29]:

system_prompt = (
    "You are an Medical assistant named HeathMateAI for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [30]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [31]:
#test
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder caused by the abnormal release of a chemical from the pituitary gland, leading to increased growth of bone and soft tissue along with various other disturbances, occurring after bone growth has stopped. Gigantism, on the other hand, occurs when this abnormality happens before the end of bone growth, leading to unusual height. Both conditions are related to excess growth hormone activity.


In [32]:
response = rag_chain.invoke({"input": "what is Acne?"})
print(response["answer"])

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria. The medical term for common acne is acne vulgaris.


In [33]:
response = rag_chain.invoke({"input": "what is the Treatment of Acne?"})
print(response["answer"])

The treatment for acne depends on its severity. Mild noninflammatory acne can be treated with topical medications like tretinoin, benzoyl peroxide, adapalene, or salicylic acid. For inflammatory acne, topical antibiotics may be added, with improvement typically seen in two to four weeks, and severe cases may require oral medications such as isotretinoin.
